# Task 1.1 — Read the file the foundations way

In [ ]:
from csv import DictReader
import json
import csv

In [ ]:
input_file = "/content/appointments.csv"
output_file = "/content/appointments_clean.csv"

def parse_row(row):
  return{
      "appointment_id": row["appointment_id"],
      "patient_id": row["patient_id"],
      "clinic_id": row["clinic_id"],
      "booked_on": row["booked_on"],
      "appt_date": row["appt_date"],
      "age": int(row["age"]),
      "gender": row["gender"],
      "distance_km": float(row["distance_km"]),
      "fee_inr": int(row["fee_inr"]),
      "booking_channel": row["booking_channel"],
      "visit_type": row["visit_type"],
      "reminder_sent": row["reminder_sent"],
      "prior_no_shows": int(row["prior_no_shows"]),
      "status": row["status"]
    }

accepted_rows =[]
rejected_rows =[]
row_count = 0

with open(input_file, "r", newline="")as file:
    reader = csv.DictReader(file)
    print("Header:", reader.fieldnames)

    for row in reader:
      row_count +=1
      try:
          parsed = parse_row(row)
          accepted_rows.append(parsed)
      except ValueError:
          rejected_rows.append(row)

print(f"Total_rows: {row_count}")

print(f"Accepted: {len(accepted_rows)}")

print(f"Rejected: {len(rejected_rows)}")

with open("/content/appointments_clean.csv", "w", newline ="") as json_file:
  json.dump(rejected_rows, json_file, indent=2)



Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total_rows: 1020
Accepted: 988
Rejected: 32


# Task 1.2 — Inspect and clean

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#Load the file with pd.read_csv. Print shape, info(), isnull().sum() and duplicated().sum().
#Then, in this order:
#  (a) drop exact duplicate rows;
 # (b) clean booking_channel with .str.strip().str.title();
  #(c) convert appt_date with pd.to_datetime(..., errors="coerce") and booked_on with pd.to_datetime;
  #(d) drop rows where appt_date is missing after conversion or status is missing.
#Print the row count after each step. This task starts again from the original appointments.csv:
#Task 1.1 is a separate exercise in reading with the csv module, so do not use its accepted/rejected lists here — the pandas cleaning in 1.2–1.3 is
#the one that feeds the rest of the project.



In [ ]:
df = pd.read_csv("/content/appointments.csv")

In [ ]:
df.shape

(1020, 14)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB


In [ ]:
df.isnull().sum()

,0
appointment_id,0
patient_id,0
clinic_id,0
booked_on,0
appt_date,1
age,12
gender,0
distance_km,20
fee_inr,0
booking_channel,0


In [ ]:
df.duplicated().sum()


np.int64(20)

In [ ]:
df.head()

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status
0,A0001,P509,C05,2025-01-24,2025-01-25,60.0,F,18.0,850,Phone,Consultation,Yes,1,No-Show
1,A0002,P046,C07,2025-05-27,2025-06-06,39.0,M,12.5,350,Walk-in,Follow-up,No,2,Attended
2,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
3,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
4,A0004,P482,C01,2025-04-18,2025-04-22,52.0,F,7.5,750,App,Consultation,Yes,0,Attended


In [ ]:
# drop exact duplicate rows;
df = df.drop_duplicates()
print("After drop duplicate values:", len(df))


After drop duplicate values: 1000


In [ ]:
# clean booking_channel
df['booking_channel'] = df['booking_channel'].str.strip().str.title()           # Remove whitespace and format text
print("After clean booking_channel:", len(df))

After clean booking_channel: 1000


In [ ]:
# convert datetime
df['appt_date']= pd.to_datetime(df['appt_date'],errors ='coerce')
df['booked_on']= pd.to_datetime(df['booked_on'])
print("After date conversion:",len(df))

After date conversion: 1000


In [ ]:
# Drop missing rows:
df = df.dropna(subset=['appt_date','status'])
print("After drop missing rows:",len(df))

After drop missing rows: 990


Task 1.3 — Clean, pass 2: validity, gaps and outliers (5 marks)
Work in this order, so that the quartiles and fences in (c) are computed after the gaps are filled.
 (a) Treat any age below 1 or above 100 as invalid and set it to missing.
 (b) Fill missing age with the median (then .round().astype(int)) and missing distance_km with its median.
 (c) For distance_km and fee_inr, compute Q1, Q3 and IQR with .quantile, and cap values at Q3 + 1.5×IQR and Q1 − 1.5×IQR with np.clip.
 (d) Round distance_km to 1 decimal and fee_inr to whole rupees (.round().astype(int)). Print how many values you capped in each column.

# Task 1.3 — Clean, pass 2: validity, gaps and outliers


In [ ]:
# Remove invalid age
df.loc[(df['age']<1)|(df['age']>100),"age"] = np.nan

In [ ]:
# Fill missing values in "age column","distance_km":
df["age"]= df["age"].fillna(df["age"].median()).round().astype(int)
df["distance_km"]= df["distance_km"].fillna(df["distance_km"].median())


In [ ]:
# (c) Compute Q1, Q3, IQR and cap outliers using np.clip for distance_km and fee_inr
# 1. distance_km
q1_distance =df['distance_km'].quantile(0.25)
q3_distance =df['distance_km'].quantile(0.75)
iqr = q3_distance - q1_distance
lower_distance = q1_distance -1.5 * iqr
upper_distance =q3_distance + 1.5 * iqr

# Count how many values fall outside the fences before clipping
capped_distance = ((df['distance_km']< lower_distance)|(df['distance_km']> upper_distance)).sum()
df['distance_km'] = np.clip(df['distance_km'], lower_distance, upper_distance)
print(" distance: q1_distance, q3_distance, upper_distance:",len(df))

# 2. fee_inr
q1_fee = df['fee_inr'].quantile(0.25)
q3_fee = df['fee_inr'].quantile(0.75)
iqr = q3_fee - q1_fee
lower_fee = q1_fee -1.5 * iqr
upper_fee = q3_fee + 1.5 * iqr

# Count how many values fall outside the fences before clipping
capped_fee =((df['fee_inr']< lower_fee)|(df['fee_inr']>upper_fee)).sum()
df['fee_inr'] = np.clip(df['fee_inr'],lower_fee, upper_fee)
print("fee: q1_fee, q3_fee, upper_fee:",len(df))


 distance: q1_distance, q3_distance, upper_distance: 990
fee: q1_fee, q3_fee, upper_fee: 990


In [ ]:
# (d) Round distance_km to 1 decimal and fee_inr to whole rupees, and print capped counts
df['distance_km'] = df['distance_km'].round(1)
df['fee_inr'] = df['fee_inr'].round().astype(int)

print("Values capped in distance_km:", capped_distance)
print("Values capped in fee_inr:",capped_fee)

Values capped in distance_km: 26
Values capped in fee_inr: 3


# Task 1.4 — Features and a NumPy summary

> Add blockquote



In [ ]:
# Create lead_days by subtracting booking date from appointment date
df['lead_days'] =(df['appt_date']-df['booked_on']).dt.days                          #.dt.days-used to extract the integer number of days
                                                                                    # sequence required for subtraction
df['no_show'] = np.where(df['status']=="No-Show",1,0)                               # for converting text data into binary target
                                                                                    # np.where - work like if...else statement

print("Lead days runs from:", df['lead_days'].min(), "to", df['lead_days'].max())
print("Total no-shows:", df['no_show'].sum(), "out of", len(df))

Lead days runs from: 1 to 45
Total no-shows: 218 out of 990


In [ ]:
# a) Calculate the column-wise mean for age, distance_km, fee_inr, and lead_days
X = df[["age","distance_km","fee_inr","lead_days"]].to_numpy()
print("Mean of each column:",X.mean(axis=0))


Mean of each column: [ 48.17373737   9.50090909 635.71414141  11.81919192]


In [ ]:
#b) Create a boolean mask to count and calculate the percentage share of appointments with lead_days > 14mask = df['lead_days']>14
mask = X[:, 3] > 14
count_gt_14 = mask.sum()
share_gt_14 = mask.mean()

#print("Number of rows with lead_days > 14:", count_gt_14)
#print(f"Share of rows with lead_days > 14:, {share_gt_14 * 100:.1f}%")

print(f"Number of rows: {count_gt_14}, Share: {share_gt_14 * 100:.1f}%")

Number of rows: 296, Share: 29.9%


In [ ]:
# c) Z-scores of distance_km (column index 1) and count with |z| > 2
x= X[:,1]
z_scores =(x-np.mean(x))/np.std(x)
count_z =(np.abs(z_scores)>2).sum()
print(f"{count_z} rows have |z| >2 for distance_km")

53 rows have |z| >2 for distance_km


# Task 1.5 — Enrich the clinics from a live API

In [ ]:
import pandas as pd
import requests

In [ ]:
clinics = pd.read_csv("/content/clinics.csv")

In [ ]:
clinics.head()

,clinic_id,clinic_name,city,tier,latitude,longitude
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873


In [ ]:
url = "https://api.open-meteo.com/v1/forecast"                                  # Calling requests.get() alone fails because it needs city coordinates (params) and a timeout to avoid hanging
temperatures = []

# Run a loop for every clinics
for index,row in clinics.iterrows():
    params ={
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    temp_c = None
    reason = ""

    for attempt in range(2):
        try:
            response =requests.get("https://api.open-meteo.com/v1/forecast", params=params,timeout=10)

            if response.status_code == 200:
                data =response.json()
                if "hourly" in data and "temperature_2m" in data["hourly"]:
                    temp_c = data["hourly"]["temperature_2m"][0]
                    break
                else:
                    reason = "hourly is missing"

            else:
                reason =f"Status Code: {response.status_code}"


        except requests.exceptions.RequestException as e:
                reason = f"Request Error: {e}"

    if temp_c is None:
      print(f"Failed for {row["clinic_name"]}: {reason}")

    temperatures.append(temp_c)


clinics["temp_c"] = temperatures
clinics.to_csv("/content/clinics_enriched.csv", index=False)

clinics


,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,19.2
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,19.5
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,19.8
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,16.4
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,16.4
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085,25.8
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937,16.4
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462,14.4


# Task 1.6 — Export the clean table and write the SQL load script

In [ ]:
# 1. Sort data by appointment_id and format date columns properly
df = df.sort_values("appointment_id").reset_index(drop=True)

df["booked_on"] = pd.to_datetime(df["booked_on"]).dt.strftime("%Y-%m-%d")
df["appt_date"] = pd.to_datetime(df["appt_date"]).dt.strftime("%Y-%m-%d")

# Ensure exact 16 columns order required by the task
expected_columns = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status", "lead_days", "no_show"
]
df = df[expected_columns]

# 2. Safety check: Fill any unexpected remaining missing values just in case
df["age"] = df["age"].fillna(df["age"].median()).astype(int)
df["distance_km"] = df["distance_km"].fillna(df["distance_km"].median())

# 3. Validation checks
assert df["appointment_id"].is_unique, "appointment_id is not unique!"
#assert df.isnull().sum().sum(),"There are missing values in the dataframe!"
print("Validation successful: appointment_id is unique.")

# 4. Export the clean dataframe to CSV
df.to_csv("/content/appointments_clean.csv", index=False)
print("Cleaned data saved to appointments_clean.csv")

# 5. Generate the load_data.sql file using f-strings
clinics_enriched = pd.read_csv("/content/clinics_enriched.csv")

with open("/content/load_data.sql", "w") as f:
    # First, write INSERT queries for clinics (required due to foreign key)
    for _, row in clinics_enriched.iterrows():
        temp_val = "NULL" if pd.isna(row["temp_c"]) else row["temp_c"]
        f.write(
            f"INSERT INTO clinics VALUES ('{row['clinic_id']}', "
            f"'{row['clinic_name']}', '{row['city']}', '{row['tier']}', "
            f"{row['latitude']}, {row['longitude']}, {temp_val});\n"
        )

    # Second, write INSERT queries for appointments
    for _, row in df.iterrows():
        f.write(
            f"INSERT INTO appointments VALUES ('{row['appointment_id']}', "
            f"'{row['patient_id']}', '{row['clinic_id']}', '{row['booked_on']}', "
            f"'{row['appt_date']}', {row['age']}, '{row['gender']}', "
            f"{row['distance_km']}, {row['fee_inr']}, '{row['booking_channel']}', "
            f"'{row['visit_type']}', '{row['reminder_sent']}', {row['prior_no_shows']}, "
            f"'{row['status']}', {row['lead_days']}, {row['no_show']});\n"
        )

print("load_data.sql file has been successfully generated and saved!")

Validation successful: appointment_id is unique.
Cleaned data saved to appointments_clean.csv
load_data.sql file has been successfully generated and saved!
